In [1]:
%pip -q install "datasets==3.6.0" torchaudio tqdm
import datasets
print("datasets:", datasets.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.1 MB/s eta 0:00:00
datasets: 3.6.0


In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
HF_CACHE = "/content/drive/MyDrive/hf_cache"
os.makedirs(HF_CACHE, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE
os.environ["HF_DATASETS_CACHE"] = os.path.join(HF_CACHE, "datasets")
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE, "hub")

print("HF cache:", HF_CACHE)


Mounted at /content/drive
HF cache: /content/drive/MyDrive/hf_cache


In [3]:
import os
import torch
import torchaudio
from tqdm import tqdm
from datasets import load_dataset, interleave_datasets, DatasetDict


In [4]:
SOUTH_ASIAN_LANGS = [
    "Assamese", "Bengali", "Gujarati", "Hindi", "Kannada", "Malayalam",
    "Marathi", "Nepali", "Oriya", "Punjabi", "Sindhi", "Tamil", "Telugu", "Urdu"
]

CANDIDATES = {
    "Assamese": ["as_in"],
    "Bengali":  ["bn_in", "bn_bd"],
    "Gujarati": ["gu_in"],
    "Hindi":    ["hi_in"],
    "Kannada":  ["kn_in"],
    "Malayalam":["ml_in"],
    "Marathi":  ["mr_in"],
    "Nepali":   ["ne_np", "ne_in"],
    "Oriya":    ["or_in"],                 # sometimes shown as Odia/Oriya
    "Punjabi":  ["pa_in", "pa_pk"],
    "Sindhi":   ["sd_in", "sd_pk"],
    "Tamil":    ["ta_in"],
    "Telugu":   ["te_in"],
    "Urdu":     ["ur_pk", "ur_in"],
}

def normalize_lang(s):
    return (s or "").strip().lower()

def resolve_config_for_language(lang_name, candidates):
    for cfg in candidates:
        try:
            ds = load_dataset("google/fleurs", cfg, split="train", streaming=True, trust_remote_code=True)
            ex = next(iter(ds))
            ex_lang = ex.get("language", "")
            if lang_name == "Oriya":
                # accept either spelling
                if "odia" in normalize_lang(ex_lang) or "oriya" in normalize_lang(ex_lang):
                    return cfg
            else:
                if normalize_lang(ex_lang) == normalize_lang(lang_name):
                    return cfg
        except Exception:
            pass
    raise RuntimeError(f"Could not resolve config for {lang_name}. Tried: {candidates}")

south_asian_configs = []
for lang in SOUTH_ASIAN_LANGS:
    cfg = resolve_config_for_language(lang, CANDIDATES[lang])
    south_asian_configs.append(cfg)
    print(f"{lang:10s} -> {cfg}")

print("\nConfigs:", south_asian_configs)
print("Num configs:", len(south_asian_configs))


README.md: 0.00B [00:00, ?B/s]

fleurs.py: 0.00B [00:00, ?B/s]

Assamese   -> as_in
Bengali    -> bn_in
Gujarati   -> gu_in
Hindi      -> hi_in
Kannada    -> kn_in
Malayalam  -> ml_in
Marathi    -> mr_in
Nepali     -> ne_np
Oriya      -> or_in
Punjabi    -> pa_in
Sindhi     -> sd_in
Tamil      -> ta_in
Telugu     -> te_in
Urdu       -> ur_pk

Configs: ['as_in', 'bn_in', 'gu_in', 'hi_in', 'kn_in', 'ml_in', 'mr_in', 'ne_np', 'or_in', 'pa_in', 'sd_in', 'ta_in', 'te_in', 'ur_pk']
Num configs: 14


In [5]:
global_ids = []
for cfg in south_asian_configs:
    ds = load_dataset("google/fleurs", cfg, split="train", streaming=True, trust_remote_code=True)
    ex = next(iter(ds))
    global_ids.append(int(ex["lang_id"]))

global_to_local = {gid: i for i, gid in enumerate(global_ids)}
num_classes = len(global_ids)

print("Global lang_ids:", global_ids)
print("Num classes:", num_classes)
print("Example mapping:", list(global_to_local.items())[:5])


Global lang_ids: [3, 8, 29, 32, 47, 59, 61, 66, 72, 73, 79, 87, 88, 94]
Num classes: 14
Example mapping: [(3, 0), (8, 1), (29, 2), (32, 3), (47, 4)]


In [6]:
def load_group_dataset_non_streaming(configs):
    ds_list = []
    for cfg in configs:
        print("Loading config:", cfg)
        ds_list.append(load_dataset("google/fleurs", cfg, trust_remote_code=True))

    combined = DatasetDict({
        split: interleave_datasets([ds[split] for ds in ds_list])
        for split in ds_list[0].keys()
    })

    combined["train"] = combined["train"].shuffle(seed=42)
    combined["validation"] = combined["validation"].shuffle(seed=42)
    return combined

sa_ds = load_group_dataset_non_streaming(south_asian_configs)
print(sa_ds)


Loading config: as_in


data/as_in/audio/train.tar.gz:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

data/as_in/audio/dev.tar.gz:   0%|          | 0.00/267M [00:00<?, ?B/s]

data/as_in/audio/test.tar.gz:   0%|          | 0.00/656M [00:00<?, ?B/s]

train.tsv: 0.00B [00:00, ?B/s]

...

test.tsv: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 15134
    })
    validation: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 3346
    })
    test: Dataset({
        features: ['id', 'num_samples', 'path', 'audio', 'transcription', 'raw_transcription', 'gender', 'lang_id', 'language', 'lang_group_id'],
        num_rows: 4186
    })
})


In [7]:
TARGET_SR = 16000
N_MELS = 80

mel = torchaudio.transforms.MelSpectrogram(
    sample_rate=TARGET_SR, n_fft=400, hop_length=160, n_mels=N_MELS
)
amp_to_db = torchaudio.transforms.AmplitudeToDB()

def waveform_to_logmel(waveform_1xN: torch.Tensor) -> torch.Tensor:
    m = mel(waveform_1xN)                          # [1,80,T]
    lm = amp_to_db(m)                              # [1,80,T]
    return lm.squeeze(0).transpose(0, 1).contiguous()  # [T,80]


In [8]:
SAVE_ROOT = "/content/drive/MyDrive/fleurs_preprocessed/south_asian"
os.makedirs(SAVE_ROOT, exist_ok=True)

def preprocess_and_save_shards(split_ds, shard_dir, prefix, shard_size=500):
    os.makedirs(shard_dir, exist_ok=True)
    shard, shard_idx, total = [], 0, 0

    for ex in tqdm(split_ds):
        wav = torch.tensor(ex["audio"]["array"], dtype=torch.float32)
        gid = int(ex["lang_id"])
        lid = global_to_local[gid]

        x = waveform_to_logmel(wav.unsqueeze(0))  # [T,80]

        shard.append({"x": x, "y": lid, "global_lang_id": gid, "language": ex.get("language","N/A")})
        total += 1

        if len(shard) >= shard_size:
            out = os.path.join(shard_dir, f"{prefix}_shard_{shard_idx:03d}.pt")
            torch.save(shard, out)
            shard, shard_idx = [], shard_idx + 1

    if shard:
        out = os.path.join(shard_dir, f"{prefix}_shard_{shard_idx:03d}.pt")
        torch.save(shard, out)

    print(f"Saved {total} examples -> {shard_dir}")

train_shards = os.path.join(SAVE_ROOT, "train_shards")
val_shards   = os.path.join(SAVE_ROOT, "validation_shards")
test_shards  = os.path.join(SAVE_ROOT, "test_shards")

preprocess_and_save_shards(sa_ds["train"], train_shards, "train", shard_size=500)
preprocess_and_save_shards(sa_ds["validation"], val_shards, "validation", shard_size=500)
preprocess_and_save_shards(sa_ds["test"], test_shards, "test", shard_size=500)


100%|██████████| 15134/15134 [05:43<00:00, 44.08it/s]


Saved 15134 examples -> /content/drive/MyDrive/fleurs_preprocessed/south_asian/train_shards


100%|██████████| 3346/3346 [01:04<00:00, 52.02it/s]


Saved 3346 examples -> /content/drive/MyDrive/fleurs_preprocessed/south_asian/validation_shards


100%|██████████| 4186/4186 [01:27<00:00, 47.63it/s]


Saved 4186 examples -> /content/drive/MyDrive/fleurs_preprocessed/south_asian/test_shards
